In [2]:
"""
COMMENT 6 - 95% CI for the propensity-matched AST AUC (0.760).

"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ---- paths (already correct for your machine; edit only if moved) ----
ROOT        = Path("/data0/b2ai-voice/3.0.0")
DEMO_PATH   = ROOT / "phenotype" / "demographics" / "demographics.tsv"

NPZ         = Path.cwd() / "ast_pd_v3_cv_results.npz"

# ---- 1) load SAVED OOF predictions (no model, no GPU) ----
d   = np.load(NPZ, allow_pickle=True)
pid = np.array([str(p).zfill(6) for p in d["participant_ids"]])
ast = d["oof_probs"].astype(float)
lab = d["oof_labels"].astype(int)
base = pd.DataFrame({"participant_id": pid, "label": lab, "ast": ast})

# ---- 2) demographics (identical coercion to cell #15) ----
demo = pd.read_csv(DEMO_PATH, sep="\t")
demo["participant_id"] = demo["participant_id"].astype(str).str.zfill(6)
demo["age"] = pd.to_numeric(demo["age"].replace({"90 and above": 90}), errors="coerce")
demo = demo.sort_values("age", na_position="last").groupby("participant_id").first().reset_index()
df = base.merge(demo[["participant_id", "age", "sex_at_birth", "country"]],
                on="participant_id", how="left")
df["sex"] = (df["sex_at_birth"] == "Male").astype(int)

# ---- 3) USA-only ages 60-80 subgroup ----
cty = df["country"].astype(str)
usa = cty.str.upper().str.startswith("US") | (cty.str.lower() == "united states")
sub = df[(df["age"] >= 60) & (df["age"] <= 80) & usa].dropna(subset=["age", "sex"]).reset_index(drop=True)
print(f"subgroup N={len(sub)}  cases={int(sub.label.sum())}  ctrl={int((sub.label==0).sum())}")

# ---- 4) propensity + greedy 1:1 nearest-neighbor match (seed 42, as in cell #15) ----
lr = LogisticRegression(class_weight="balanced", max_iter=1000).fit(sub[["age", "sex"]].values, sub["label"].values)
sub["prop"] = lr.predict_proba(sub[["age", "sex"]].values)[:, 1]
cases = sub[sub.label == 1].reset_index(drop=True)
ctrls = sub[sub.label == 0].reset_index(drop=True)
caliper = 0.2 * sub["prop"].std()
used, matches = set(), []
rng = np.random.default_rng(42)
for ci in rng.permutation(len(cases)):
    c = cases.iloc[ci]
    cand = [j for j in range(len(ctrls)) if j not in used]
    if not cand:
        break
    dists = np.array([abs(c["prop"] - ctrls.iloc[j]["prop"]) for j in cand])
    b = int(np.argmin(dists))
    if dists[b] > caliper:
        continue
    used.add(cand[b])
    matches.append((c, ctrls.iloc[cand[b]]))
matched = pd.DataFrame([m[0] for m in matches] + [m[1] for m in matches])
matched_auc = roc_auc_score(matched.label, matched.ast)
print(f"matched pairs={len(matches)}  matched AST AUC={matched_auc:.4f}")

# ---- 5) bootstrap 95% CI for the matched AST AUC ----
y, p = matched.label.values, matched.ast.values
rb, boot = np.random.default_rng(42), []
for _ in range(2000):
    idx = rb.integers(0, len(y), len(y))
    if len(np.unique(y[idx])) < 2:
        continue
    boot.append(roc_auc_score(y[idx], p[idx]))
lo, hi = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
print(f"\n>>> Matched AST AUC = {matched_auc:.3f}  (95% CI [{lo:.3f}, {hi:.3f}])  <<<")
print("    (analytic Hanley-McNeil cross-check was ~[0.634, 0.887])")

# ---- 6) write the CI back into the results JSON ----
for jp in [RESULTS_DIR / "delong_and_propensity_pd.json",
           Path("new_delong_and_propensity_pd.json")]:
    if jp.exists():
        J = json.load(open(jp))
        J.setdefault("propensity", {})["matched_auc_ci"] = [lo, hi]
        json.dump(J, open(jp, "w"), indent=2)
        print("updated:", jp)


subgroup N=84  cases=39  ctrl=45
matched pairs=28  matched AST AUC=0.7602

>>> Matched AST AUC = 0.760  (95% CI [0.628, 0.877])  <<<
    (analytic Hanley-McNeil cross-check was ~[0.634, 0.887])
